In [1]:
#installing necessary packages
import os
import pandas as pd
import numpy as np
import glob

import matplotlib.pyplot as plt

from astroquery.esa.euclid.core import Euclid
from astroquery.simbad import Simbad

import astropy.units as u
from astropy import wcs
from astropy.nddata.utils import Cutout2D
from astropy.coordinates import SkyCoord
from astropy.table import Table
from astropy.io import fits
from astropy.visualization import astropy_mpl_style, ImageNormalize, PercentileInterval, AsinhStretch, LogStretch

plt.style.use(astropy_mpl_style)

In [2]:
#making a folder for the output files
output_folder= '/media/user/output2/'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [3]:
Euclid.login(credentials_file='/media/user/cred.txt')

INFO: Login to Euclid TAP server: eas.esac.esa.int:443/tap-server/tap/ [astroquery.esa.euclid.core]
INFO: OK [astroquery.utils.tap.core]
INFO: Login to Euclid data service: eas.esac.esa.int:443/sas-dd/tap-server/ [astroquery.esa.euclid.core]
INFO: OK [astroquery.utils.tap.core]
INFO: Login to Euclid cutout service: eas.esac.esa.int:443/sas-cutout/tap-server/ [astroquery.esa.euclid.core]
INFO: OK [astroquery.utils.tap.core]


## Important Euclid tables
- **catalogue.mer_catalogue**: 

    Contains basic galaxy information, i.e. right ascension, declination, ellipticity, fwhm, etc...
    
    Information available at: https://euclid.esac.esa.int/dr/q1/dpdd/merdpd/dpcards/mer_finalcatalog.html#main-catalog-fits-file
- **catalogue.mer_morphology**: 

    Contains zoobot classifications
    
    Information available at: https://euclid.esac.esa.int/dr/q1/dpdd/merdpd/dpcards/mer_finalcatalog.html#morphology-catalog-fits-file
- **catalogue.phz_physical_parameters**: 

    Contains redshift and magnitude data
    
    Information available at: https://euclid.esac.esa.int/dr/q1/dpdd/phzdpd/dpcards/phz_phzpfoutputforl3.html#physical-parameters-catalog

In [ ]:
#Lists all publically available Euclid tables
tables = Euclid.load_tables(only_names=True, include_shared_tables=True)
for table in (tables):
    print(table.get_qualified_name())

In [ ]:
#Lists values available within each Euclid table
catalogue = "catalogue.mer_morphology"

table = Euclid.load_table(table=catalogue, verbose=False)
for column in table.columns:
    print(f'{column.name}')
    

## Simple Query

- Start by choosing which catalogues you need from the Euclid tables
    
- If you are selecting specific values to request then to join multiple tables use:

    JOIN table AS name USING(object_id)
    
    
- If you are selecting all values from the tables with "SELECT \*" then you may need to use:

    JOIN table AS name USING(object_id, to_be_published, basic_download_data_oid)
    
- WHERE statement behaves interestingly and it has, on multiple occasions, changed what functions can be present within it for me.

    If you wish to limit your range of, for example, kron_radius to greater than 200 you need to use:
    
        WHERE mer.kron_radius BETWEEN 200 AND 1e+15
    
    If an object doesn't have a value it will be a masked value which will appear as "--" and will be filled behind the scenes as 1e+20. THIS IS NOT A NULL VALUE AND CANNOT BE TREATED AS SUCH.
    
    To filter out these value you need to clamp your values between your desired value and a value less than 1e+20 using BETWEEN or a less than operation.
    
- ORDER BY seems to work like normal SQL and you can use both ASC and DESC

## DEBUGGING

- The launch_job_async offers little to no useful information if your query fails, the only way to try and get any useful information about why it failed is to enable verbose.

    This will provide a link which occasionally lets you access a more in-depth reason as to why your query failed.
    
    Most of the time it will tell you that you do not have access to the link you clicked.
    
    I find logging into the ESA datalabs website helps with this but sometimes it tells me I don't have authorisation to log in, so good luck.
    

In [6]:
#This is a basic query for identifying the top 10 objects with kron radii greater than 200 and returning the RA, DEC, Kron radius, and Ellipticity of the object.

query = """
    SELECT TOP 10 mer.object_id, mer.right_ascension, mer.declination, mer.kron_radius, mer.ellipticity

    FROM catalogue.mer_catalogue AS mer
    
    WHERE mer.kron_radius BETWEEN 200 AND 1e+15
    
    ORDER BY mer.kron_radius DESC
"""

#Launches the query
job = Euclid.launch_job_async(query, verbose=True)
result = job.get_results()
#Outputs the result of the query
result

Launched query: '
    SELECT TOP 10 mer.object_id, mer.right_ascension, mer.declination, mer.kron_radius, mer.ellipticity

    FROM catalogue.mer_catalogue AS mer

    WHERE mer.kron_radius BETWEEN 200 AND 1e+15

    ORDER BY mer.kron_radius DESC
'
------>https
host = eas.esac.esa.int:443
context = /tap-server/tap/async
Content-type = application/x-www-form-urlencoded
303 
[('Date', 'Fri, 16 Oct 2026 02:38:54 GMT'), ('Server', 'Apache/2.4.37 (Red Hat Enterprise Linux) OpenSSL/1.1.1k mod_jk/1.2.49'), ('X-VO-Authenticated', 'mlayla02'), ('Location', 'https://eas.esac.esa.int/tap-server/tap/async/baaac1c6-c90a-11f1-a989-e8ebd3edb7d7-TPDR'), ('X-Content-Type-Options', 'nosniff'), ('X-XSS-Protection', '0'), ('Cache-Control', 'no-cache, no-store, max-age=0, must-revalidate'), ('Pragma', 'no-cache'), ('Expires', '0'), ('X-Frame-Options', 'SAMEORIGIN'), ('Content-Type', 'text/plain;charset=UTF-8'), ('Vary', 'Accept-Encoding'), ('Transfer-Encoding', 'chunked')]
job baaac1c6-c90a-11f1-a989-e8ebd

object_id,right_ascension,declination,kron_radius,ellipticity
NA,deg,deg,pix,NA
int64,float64,float64,float64,float64
-855952154080940138,85.59521545024064,-8.094013824256796,20723.599609375,0.28035226464271545
-603342520504810458,60.334252056047355,-50.481045851456976,16941.79296875,0.6359493136405945
2738274659685969936,273.8274659780114,68.59699361368902,16190.5830078125,0.686718225479126
-857590885083502074,85.75908854811767,-8.3502074280046,16077.72265625,0.7523468136787415
-619489843501452337,61.94898435197474,-50.14523378208134,15925.908203125,0.5606241226196289
-857860610085390917,85.78606100313111,-8.539091732324415,15475.1064453125,0.3329198956489563
2650725720671827114,265.0725720458137,67.18271143961226,14977.958984375,0.9783209562301636
2665349906676249555,266.53499062283566,67.62495555926346,14579.232421875,0.16728635132312775


## Complex queries

- You may want to perform more complex queries such as only using barred galaxies. This can be done by using sub queries.
    
- With Sub Queries you can define new variables from existing ones and use them to filter out more specific details

- For example defining a bar score which is the a combination of the 3 bar strength values then using that to filter only barred galaxies.
    

In [7]:
query = """
SELECT TOP 1000 * 

    FROM (
      SELECT mor.object_id, mer.segmentation_map_id,
      
          mor.bar_strong, mor.bar_weak, mor.bar_no,
          
          mer.right_ascension, mer.declination, mer.kron_radius, mer.ellipticity, mer.semimajor_axis, mer.fwhm,
          
          phz.phz_pp_median_redshift, phz.phz_pp_mode_redshift,
          phz.phz_pp_median_stellarmass, phz.phz_pp_mode_stellarmass,
  
          (mor.bar_strong + mor.bar_weak - mor.bar_no)  AS bar_score,
          DEGREES(ACOS(1 - mer.ellipticity)) AS inclination        
  
      FROM catalogue.mer_morphology AS mor
      JOIN catalogue.mer_catalogue AS mer USING(object_id)
      JOIN catalogue.phz_physical_parameters AS phz USING(object_id)
  
    ) AS sub
    
WHERE 
bar_score > 0 AND bar_score < 1e+15 AND 
inclination > 40 AND inclination < 70 AND
phz_pp_median_redshift < 0.2

ORDER BY bar_score DESC
"""
#AND phz_pp_median_redshift < 0.2 
#(mer.semimajor_axis / mer.fwhm) AS resolution
#/ (mor.bar_strong + mor.bar_weak + mor.bar_no))
job = Euclid.launch_job_async(query, verbose=True)
result = job.get_results()
#Outputs the result of the query
result


Launched query: '
SELECT TOP 1000 * 

    FROM (
      SELECT mor.object_id, mer.segmentation_map_id,

          mor.bar_strong, mor.bar_weak, mor.bar_no,

          mer.right_ascension, mer.declination, mer.kron_radius, mer.ellipticity, mer.semimajor_axis, mer.fwhm,

          phz.phz_pp_median_redshift, phz.phz_pp_mode_redshift,
          phz.phz_pp_median_stellarmass, phz.phz_pp_mode_stellarmass,

          (mor.bar_strong + mor.bar_weak - mor.bar_no)  AS bar_score,
          DEGREES(ACOS(1 - mer.ellipticity)) AS inclination        

      FROM catalogue.mer_morphology AS mor
      JOIN catalogue.mer_catalogue AS mer USING(object_id)
      JOIN catalogue.phz_physical_parameters AS phz USING(object_id)

    ) AS sub

WHERE 
bar_score > 0 AND bar_score < 1e+15 AND 
inclination > 40 AND inclination < 70 AND
phz_pp_median_redshift < 0.2

ORDER BY bar_score DESC
'
------>https
host = eas.esac.esa.int:443
context = /tap-server/tap/async
Content-type = application/x-www-form-urlencoded
303

inclination,right_ascension,fwhm,bar_no,phz_pp_mode_stellarmass,bar_score,object_id,semimajor_axis,phz_pp_median_stellarmass,bar_weak,declination,bar_strong,phz_pp_mode_redshift,segmentation_map_id,phz_pp_median_redshift,kron_radius,ellipticity
,deg,arcsec,NA,NA,,NA,pix,NA,NA,deg,NA,NA,NA,NA,pix,NA
float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64
43.462708442666916,54.58031471416895,1.207850456237793,2.015383005142212,10.136415481567383,135.06146788597107,-545803147272089049,23.914058685302734,10.136415481567383,45.46382141113281,-27.20890494177033,91.61302947998047,0.16022692620754242,102045470003728,0.16022692620754242,167.39840698242188,0.27417775988578796
53.250729094503484,61.994039259806954,1.3352792263031006,3.114027738571167,10.29382610321045,133.91301083564758,-619940392473379873,24.93345832824707,10.29382610321045,52.825599670410156,-47.33798735969931,84.2014389038086,0.16304191946983337,102021986080269,0.16304191946983337,174.53421020507812,0.40168559551239014
66.12262294658724,51.182411369670504,1.1470180749893188,5.223933696746826,8.31706428527832,128.52156114578247,-511824113271367549,27.52452850341797,8.31706428527832,53.540279388427734,-27.136754968921476,80.20521545410156,0.024373117834329605,102045464021200,0.024373117834329605,192.67169189453125,0.5952194333076477
41.453226649829425,274.3852988969152,1.35811448097229,2.556873321533203,9.494884490966797,126.87544631958008,2743852988667923353,31.60773277282715,9.567745208740234,37.690521240234375,66.79233531333705,91.7417984008789,0.056945763528347015,102159779011431,0.059819210320711136,221.25413513183594,0.2505035996437073
49.34647478875081,265.20337938881454,1.325936198234558,6.57841682434082,9.062402725219727,125.53458595275879,2652033793642890722,13.608190536499023,9.062402725219727,56.4636344909668,64.28907223713522,75.64936828613281,0.09602825343608856,102158270010658,0.09602825343608856,95.25733947753906,0.3485167622566223
51.99404532918988,274.3360615101575,1.3840978145599365,2.5653133392333984,9.241340637207031,122.46621131896973,2743360615652429146,24.112060546875,9.241340637207031,35.263755798339844,65.24291467852774,89.76776885986328,0.07357487082481384,102158589098823,0.07357487082481384,168.784423828125,0.38425663113594055
53.840417714627286,267.90164976136265,1.3181179761886597,4.301888465881348,8.266489028930664,122.46229839324951,2679016497641998953,18.67083740234375,8.266489028930664,39.291873931884766,64.19989533202234,87.4723129272461,0.05812859162688255,102157954097314,0.05812859162688255,130.69586181640625,0.4099637269973755
41.32614651185538,59.3811661442233,1.3381938934326172,1.6320387125015259,7.716172218322754,121.63566529750824,-593811661507607832,14.907464027404785,7.716172218322754,29.718852996826172,-50.7607832269076,93.5488510131836,0.01812749356031418,102018668076149,0.01812749356031418,104.35224914550781,0.24903713166713715


In [11]:
#You can easily convert the table to a pandas array to make it easier to work with
df = result.to_pandas()
#Creates a resolution value and calculates for each galaxy what fraction of the votes goes towards the galaxy being identified as having a bar
df["resolution"] = df["semimajor_axis"] / df["fwhm"]
df["bar_yes_fraction"] = (df["bar_weak"] + df["bar_strong"]) / (df["bar_weak"] + df["bar_strong"] + df["bar_no"])

df.pop("bar_score") #Removes the "bar_score" value as it was just a temporary value used for this specific query

#Filtering out any galaxy that has a below 80% bar_yes_fraction and removes any galaxies that are too low of a resolution to ensure they are large enough to be identified, then it just sorts the dataframe based on the bar yes fraction
df = df[df["bar_yes_fraction"] > 0.8][df["resolution"] > 15].sort_values("bar_yes_fraction", ascending=False).reset_index(drop=True)
df

/tmp/ipykernel_242/1856093409.py:10: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[df["bar_yes_fraction"] > 0.8][df["resolution"] > 15].sort_values("bar_yes_fraction", ascending=False).reset_index(drop=True)


,inclination,right_ascension,fwhm,bar_no,phz_pp_mode_stellarmass,object_id,semimajor_axis,phz_pp_median_stellarmass,bar_weak,declination,bar_strong,phz_pp_mode_redshift,segmentation_map_id,phz_pp_median_redshift,kron_radius,ellipticity,resolution,bar_yes_fraction
0,43.462708,54.580315,1.207850,2.015383,10.136415,-545803147272089049,23.914059,10.136415,45.463821,-27.208905,91.613029,0.160227,102045470003728,0.160227,167.398407,0.274178,19.798857,0.985510
1,44.274331,270.897640,1.306765,2.101148,10.201084,2708976397674967682,31.443424,9.162390,27.338423,67.496768,80.520370,0.083475,102160060047868,0.029167,220.103973,0.283994,24.062039,0.980892
2,41.453227,274.385299,1.358114,2.556873,9.494884,2743852988667923353,31.607733,9.567745,37.690521,66.792335,91.741798,0.056946,102159779011431,0.059819,221.254135,0.250504,23.273246,0.980628
3,45.884120,268.324871,1.316013,2.312414,9.168896,2683248706641928224,22.665270,9.107083,38.866547,64.192822,76.587357,0.063964,102157954095664,0.067186,158.656891,0.303888,17.222680,0.980364
4,51.994045,274.336062,1.384098,2.565313,9.241341,2743360615652429146,24.112061,9.241341,35.263756,65.242915,89.767769,0.073575,102158589098823,0.073575,168.784424,0.384257,17.420778,0.979895
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,54.202900,63.580878,1.291190,18.273825,8.564109,-635808776472180220,21.327524,8.688250,15.791497,-47.218022,64.467018,0.063153,102022481008046,0.072412,149.292664,0.415083,16.517722,0.814540
220,64.835285,65.141208,1.258159,14.585785,9.639011,-651412084476551725,19.883390,9.728388,15.375880,-47.655173,48.242718,0.192643,102021990020321,0.195757,139.183731,0.574778,15.803560,0.813491
221,48.537474,52.459127,1.343505,18.278818,9.498613,-524591273289766313,21.046949,9.498613,27.162937,-28.976631,51.678741,0.190203,102042914048326,0.190203,147.328644,0.337870,15.665703,0.811792
222,68.196011,52.796917,1.322728,16.761448,9.530202,-527969167296101682,29.074181,9.530202,19.866598,-29.610168,52.317238,0.137523,102042284027228,0.129567,203.519272,0.628568,21.980475,0.811553


## Getting observations

- This function returns the vis, nisp y and nisp j images from the Q1 database

- The tile id for each observation is encoded in the segmentation_map_id variable and can be get with the following:

    tile_index = int( np.floor( segmentation_map_id / 10e+6 ) )
    
- Source: https://euclid.esac.esa.int/dr/q1/dpdd/merdpd/dpcards/mer_finalcatalog.html#detailed-description-of-the-data-product

In [12]:
def identify_observations(tile):
    vis_file = glob.glob(f'/data/euclid_q1/Q1_R1/MER/{tile}/VIS/EUC_MER_BGSUB-MOSAIC-VIS_TILE{tile}-*.fits')[0]
    nir_y_file = glob.glob(f'/data/euclid_q1/Q1_R1/MER/{tile}/NISP/EUC_MER_BGSUB-MOSAIC-NIR-Y_TILE{tile}-*.fits')[0]
    nir_j_file = glob.glob(f'/data/euclid_q1/Q1_R1/MER/{tile}/NISP/EUC_MER_BGSUB-MOSAIC-NIR-J_TILE{tile}-*.fits')[0]
    return [nir_y_file, nir_j_file, vis_file]

In [ ]:
files = []
for i,x in enumerate(df["segmentation_map_id"]):
    tile_index = int(np.floor(int(x) / 10**6))
    files.append(identify_observations(tile_index))
    
df["file_paths"] = files
df

## Create cutana DF

- Cutana needs a csv file with the following columns:

    SourceID
    
    RA
    
    DEC
    
    diameter_pixel - I'm not sure what this value actually it, nothing I put in their seems to correlate with any form of diameter, will require experimentation
    
    fits_file_paths - This needs to be in a list form like the code above provides

- Cutana needs to use a separate kernal to run and cannot be run in the standard EUCLID-TOOLS kernal

    Therefor the cutana_df needs to be saved in a csv file
    
- There are some example notebooks provided by ESA about using Cutana if you wants some more help with it.

In [ ]:
cutana_df = pd.DataFrame({
    "SourceID": df["object_id"],
    "RA": df["right_ascension"],
    "Dec": df["declination"],
    "diameter_pixel": df["kron_radius"].astype(int)/2,
    "fits_file_paths": list(df["file_paths"])
    })
cutana_df

In [62]:
cutana_df.to_csv("output/cutana_galaxies.csv", index=False)

In [6]:
df.to_csv("output/filtered_galaxies.csv", index=False)